In [ ]:
import os
import sys
import glob
import multiprocessing
import subprocess
import time
import typing
from datetime import datetime as dt
from os import path
from IPython.display import clear_output

In [ ]:
#general input parameters

available_cpu_cores: int = 30

min_dim: int = 0 #minimal problem dim
max_dim: int = 0 #maximal problem dim
cplex_tl: int = 180 #cplex time limit in minutes

# problem_lists: [str] = ["easy", "hard", "binary", "open", "feasibility", "infeasible"]
problem_lists: [str] = ["easy"]

writing_cuts_on: bool = False

In [ ]:
# ilpcuts input parameters

submat_size_lim: int = 512 #maximal size of square submat to generate conic cover cuts
row_trials_num: int = 5 #number of row trials to generate cuts
cuts_gen_rounds: int = 1 #number of cuts generation iterations
lazy_cuts_option: str = "off" #"on", "off" or "hybrid"
scaling: str = "off"
model_cuts_switch: str = "off" #switch for model cuts option
det_limit: int = 1024 #limit on maximum determinant to stard enumerative cuts generation
pureint_cuts_generation_on: bool = False
Hermite_cuts_generation_on: bool = False

In [ ]:
#internal parameters and constants

problems_name_dir: str = "../problems/benchmark/**/*.gz"
problem_lists_dir: str = "../problems/lists"

cplex_workers: int = available_cpu_cores
cuts_gen_tl: int = 0 #cuts gen time limit in minutes
node_lim: int = -1 #node limit for CPle

log_file_name_suff: str = "_log.txt" #out log file name suffix
cuts_file_name_suff: str = "_cuts.txt" #out cuts file name

out_dir_pref: str = "rootNodeSolve__"
run_path: str = "../build/" 

cuts_gen_sec_tl: int = cuts_gen_tl * 60 #cuts gen time limit in seconds
cplex_sec_tl: int = cplex_tl * 60 #cplex time limit in seconds

In [ ]:
# cut generation options

cut_generation_options: [str] = ["mir", "split"]

if pureint_cuts_generation_on : 
    cut_generation_options.append("pureint")

if Hermite_cuts_generation_on : 
    cut_generation_options.append("hermite")

split_cut_generation_options: [str] = ["gready", "fract"]

In [ ]:
# creating output directory

def form_out_dir_name():
    time_now: str = dt.now().strftime("%Y.%m.%d_%H-%M-%S")
    return out_dir_pref + time_now

out_dir_name: str = form_out_dir_name()
print(f"Output directory name: {out_dir_name}")

os.mkdir(run_path+out_dir_name)
print(f"The outpud directory ({run_path+out_dir_name}) is created!")

In [ ]:
# functions to generate run-string
def generate_run_str(file_name: str, rel_path: str, additional_run_options: typing.Dict[str,str]) -> str:

    file_options_str: str = str(list(additional_run_options.values()))
    file_options_str = file_options_str.replace(" ", "")
    add_run_options_str: str = ""

    for option_name, option_value in additional_run_options.items():
        add_run_options_str += f" {option_name} {option_value}" 

    log_rel_path: str = "./" + out_dir_name + "/" + file_name + "_" + file_options_str + "_" + log_file_name_suff 

    out_cuts_rel_path: str = "no"
    if writing_cuts_on : 
        out_cuts_rel_path = "./" + out_dir_name + "/" + file_name + "_" + file_options_str + "_" + cuts_file_name_suff

    run_str: str = f"./CPlexConsole -i {rel_path} -o {log_rel_path} --cuts_output {out_cuts_rel_path} --wcgn 1 --wcpl {cplex_workers} " \
        f"--iters {cuts_gen_rounds} --node_lim {node_lim} --model_cuts {model_cuts_switch} --trials_num {row_trials_num} --submat_limit {submat_size_lim} " \
        f"--cplex_tl {cplex_sec_tl} --cuts_tl {cuts_gen_sec_tl} --min_dim {min_dim} --max_dim {max_dim} --det_lim {det_limit}"
    
    run_str += add_run_options_str

    return run_str

def generate_baseline_run_str(file_name: str, rel_path: str) -> str:
    log_rel_path: str = "./" + out_dir_name + "/" + file_name + "_[baseline]_" + log_file_name_suff

    run_str: str = f"./CPlexConsole -b -i {rel_path} -o {log_rel_path} --wcgn 1 --wcpl {cplex_workers} " \
    f"--cplex_tl {cplex_sec_tl} --min_dim {min_dim} --max_dim {max_dim}"

    return run_str

In [ ]:
# creating the set of enabled tasks

enabled_tasks_set : typing.Set[str] = set()

for problem_list_name in problem_lists:
    list_file_name = problem_lists_dir  + "/*" + problem_list_name + "*.test"

    for rel_path in glob.iglob(list_file_name, recursive=True):
        with  open(rel_path) as file:
            enabled_tasks_set.update(file.read().splitlines()) 


# printing list of tasks to run

print(f"There are {len(enabled_tasks_set)} enabled tasks:\n")
for task_str in enabled_tasks_set:
    print(f"{task_str}")

In [ ]:
# creating list of tasks to run
tasks_to_run : [str] = []

print("Problems to work with:")

for rel_path in glob.iglob(problems_name_dir, recursive=True):
    print(rel_path)
    file_name: str = path.basename(rel_path)

    if file_name not in enabled_tasks_set:
        continue

    tasks_to_run.append(generate_baseline_run_str(file_name, rel_path))

    for cut_generation_option in cut_generation_options:
        add_options_dict: typing.Dict[str, str] = {}
        add_options_dict["--cuts_fun"] = cut_generation_option

        if cut_generation_option == "split" :
            for split_cut_option in split_cut_generation_options:
                add_options_dict["--split_mode"] = split_cut_option
                tasks_to_run.append(generate_run_str(file_name, rel_path, add_options_dict ))
        else:
            tasks_to_run.append(generate_run_str(file_name, rel_path, add_options_dict))

In [ ]:
# printing list of tasks to run

print(f"There are {len(tasks_to_run)} tasks to run:\n")
for task_str in tasks_to_run:
    print(f"{task_str}\n")

In [ ]:
#runing the tasks

def run_task(run_str : str):
    return subprocess.run(run_str, shell=True, cwd=run_path)

starting_time = dt.now()
done_tasks_counter : int = 0

for task in tasks_to_run:
    task_result = run_task(task)

    args: str = str(task_result.args)
    if task_result.returncode == 0:
        print(f"({done_tasks_counter}) The process is successfully finished!, args are:")
    else:
        print(f"({done_tasks_counter}) The process is failed!, return code is {task_result.returncode}, args are:")
    print(f"{args}\n")

    done_tasks_counter += 1
    if done_tasks_counter % 10 == 0:
        clear_output(wait=True)
        print(f"-------{done_tasks_counter} tasks are done, {len(tasks_to_run)-done_tasks_counter} in rest-------\n")

In [ ]:
# printing the duration

duration = dt.now() - starting_time
print(f"Work is finished!, duration is {duration}")